# Grad-CAM viewer — two directions, by confusion quadrant

Browse C0's saliency maps split into **TP / TN / FP / FN**.

Each image now carries **two** maps, both from the same signed Grad-CAM of the
target logit, split by sign instead of being collapsed into one:

| key | what it is |
|---|---|
| `cam_pos` = relu(signed) | regions arguing **FOR the disease** |
| `cam_neg` = relu(−signed) | regions arguing **FOR healthy** |
| `scale_pos`, `scale_neg` | each channel's pre-normalisation max |

Both channels are scaled to [0,1] by *their own* max, so a channel's colours say
where its evidence is, not how strong it is against the other channel. The raw
balance lives in `scale_pos`/`scale_neg`, and the views below report it as
`dominance = scale_pos / (scale_pos + scale_neg)` — 1.0 means the image's
gradient is entirely pro-disease, 0.0 entirely pro-healthy.

Splitting removes the orientation choice that the earlier single-map format
needed. Keeping only relu(signed) left every negative-called image all-zero (77.5%
of TN, 50.0% of FN); orienting by the predicted class instead broke the mirror
case, because `prob > threshold` is the reported *decision* while the gradient
follows the *logit*, and the two disagree between t and 0.5. With both directions
stored, nothing is annihilated, and a region that argues both ways (an opacity
with an air bronchogram through it) appears in both channels rather than
cancelling to a net near zero.

The old `cam` key is still in each file — the net map oriented toward the
reported decision — for readers written against the single-map format. Nothing in
this notebook uses it any more.

Rows whose CheXpert label is uncertain (-1) have `true = NaN` under U-Ignore. They
belong to no quadrant and are excluded throughout; previously they fell into
"incorrect" by default, because `pred == NaN` is always False.

In [69]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from PIL import Image


In [70]:
disease = "effusion"
split = "C2_dataset"
results_dir = f"/zhome/d0/a/221493/thesis/final_experiment/results/C0_final/multilabel_ignore/{disease}"
# absolute, so the notebook works from any cwd (this used to be relative and only
# resolved when the kernel happened to start at the repo root)
ths_path = os.path.join(results_dir, "threshold.txt")
cam_dir = os.path.join(results_dir, "gradcam", split)
data_dir = "/work3/s251710/thesis_data"  # THESIS_DATA root that the CSV `path` column is relative to

In [71]:
# Filenames are path.replace("/", "_") + ".npz" -- not reversible, so map CAM -> image via the C0 CSV.
# Match on basename: cam_path is absolute and may carry a different prefix than this machine.
meta = pd.read_csv(os.path.join(results_dir, f"{split}_c0_{disease}.csv"),
                   usecols=["path", "prob", "true", "correct", "certain", "cam_path"])
meta["cam_file"] = meta["cam_path"].map(os.path.basename)
meta = meta.set_index("cam_file")

ths = float(open(ths_path).read().split()[0])

# U-Ignore: a CheXpert cell of -1 (uncertain) was masked out of C0's loss, so the
# model never saw a target for it and `true` is NaN. Those rows have no quadrant
# -- `pred == true` is False against NaN, so without this filter every uncertain
# row silently lands in "incorrect" and pollutes FP/FN.
uncertain = meta["certain"] != 1
meta["pred"] = (meta["prob"] > ths).astype(int)
meta["cell"] = np.select(
    [(meta["pred"] == 1) & (meta["true"] == 1),
     (meta["pred"] == 0) & (meta["true"] == 0),
     (meta["pred"] == 1) & (meta["true"] == 0),
     (meta["pred"] == 0) & (meta["true"] == 1)],
    ["TP", "TN", "FP", "FN"], default="uncertain")

print(f"{len(meta):,} CSV rows | {len(os.listdir(cam_dir)):,} CAM files | threshold {ths:.4f}")
print(f"\n{meta['cell'].value_counts().reindex(['TP','TN','FP','FN','uncertain']).to_string()}")
print(f"\nuncertain (-1 label, excluded from every quadrant view): {int(uncertain.sum()):,}")

95,835 CSV rows | 95,835 CAM files | threshold 0.3932

cell
TP           32961
TN           39286
FP           13266
FN            5430
uncertain     4892

uncertain (-1 label, excluded from every quadrant view): 4,892


In [ ]:
# ── plotting helpers ─────────────────────────────────────────────────────────
# One definition used by every view below.
#
# Display choices, none of which touch the stored maps:
#
# 1. `vmax` is a high percentile, not the raw max. Each channel is normalised by
#    its own max, so a sparse map (one hot blob) and a broad map (evidence spread
#    over the whole chest, the usual shape for the healthy direction) get
#    stretched very differently. Clipping at a percentile makes them comparable.
#    Each channel gets its OWN percentile: sharing one vmax across the two would
#    reintroduce the joint scaling the two-map format exists to avoid.
# 2. The overlay fades out with the map value instead of being drawn at a flat
#    alpha. With a flat alpha, jet paints its lowest colour -- light blue -- over
#    every weak cell, so the whole image looks tinted. Hard-masking below a
#    cutoff fixes that but draws a sharp contour at the cutoff, which reads like
#    a segmentation boundary that Grad-CAM does not have. A graded alpha keeps
#    the map smooth: weak evidence simply fades out.
# 3. Each direction keeps a fixed colour everywhere -- pro-disease red, pro-healthy
#    blue -- so a glance at a grid tells you which way a map argues without
#    reading the title. `jet` cannot do that: it spans red AND blue within one
#    map, so the same hue would mean "strong disease evidence" in one panel and
#    "weak healthy evidence" in the next.
#
# ALPHA_GAMMA < 1 fades more gently, > 1 more aggressively. ALPHA_MAX caps the
# opacity at the hottest point so the X-ray stays visible underneath.

from matplotlib.colors import LinearSegmentedColormap

QUAD_COLOR = {"TP": "#4C9F70", "TN": "#2E6F97", "FP": "#A03B3B", "FN": "#E08A1E"}
ALPHA_MAX, ALPHA_GAMMA = 0.5, 1.6

# One sequential ramp per direction, dark -> bright, both on the same lightness
# path so neither direction looks inherently stronger than the other.
CMAP = {
    "pos": LinearSegmentedColormap.from_list("pos", ["#2B0A0A", "#B32222", "#FF7043", "#FFE0B2"]),
    "neg": LinearSegmentedColormap.from_list("neg", ["#061A2B", "#1E5FA8", "#42A5F5", "#C5E7FF"]),
}
DIR_LABEL = {"pos": "FOR disease", "neg": "FOR healthy"}
DIR_COLOR = {"pos": "#B32222", "neg": "#1E5FA8"}


def load_cam(cam_file):
    """Both directions for one image.

    Returns (cam_pos, cam_neg, scale_pos, scale_neg). Files written before the
    two-map change carry only `cam` -- rather than silently showing that net map
    in a panel labelled "FOR disease", fail loudly, because the whole point of
    these views is that the two directions are separate.
    """
    z = np.load(os.path.join(cam_dir, cam_file))
    if "cam_pos" not in z.files:
        raise KeyError(
            f"{cam_file} has no `cam_pos` -- these CAMs predate the two-direction "
            f"format (keys: {z.files}). Re-run c0_final_predictions.py.")
    return z["cam_pos"], z["cam_neg"], float(z["scale_pos"]), float(z["scale_neg"])


def dominance(s_pos, s_neg):
    """Share of the raw gradient magnitude that argues FOR the disease, in [0,1].

    The two channels are each normalised to [0,1] on disk, so this is the only
    place the balance between them survives. 0.5 means the two directions are
    equally strong; the normalised maps alone cannot tell you that.
    """
    return s_pos / (s_pos + s_neg + 1e-12)


def upsample(cam):
    """(7,7) -> (224,224), bilinear, to overlay on the X-ray."""
    return F.interpolate(torch.from_numpy(cam)[None, None], size=(224, 224),
                         mode="bilinear", align_corners=False)[0, 0].numpy()


def load_xray(path):
    return np.array(Image.open(os.path.join(data_dir, path)).convert("L").resize((224, 224)))


def _overlay(ax, img, up, vmax, cmap, alpha_max=ALPHA_MAX, gamma=ALPHA_GAMMA):
    """X-ray with one direction's CAM on top, opacity rising with the map value."""
    ax.imshow(img, cmap="gray")
    norm = np.clip(up / vmax, 0, 1)
    ax.imshow(up, cmap=cmap, vmin=0, vmax=vmax, alpha=(norm ** gamma) * alpha_max)


def show_cam(cam_file, pct=99.0, figsize=(16, 4.8)):
    """One image, both directions: X-ray, then the two overlays, then the two raw 7x7."""
    row = meta.loc[cam_file]
    cam_pos, cam_neg, s_pos, s_neg = load_cam(cam_file)
    img = load_xray(row["path"])
    dom = dominance(s_pos, s_neg)

    fig, axes = plt.subplots(1, 5, figsize=figsize)
    gt = "nan" if pd.isna(row["true"]) else f"{row['true']:.0f}"
    axes[0].imshow(img, cmap="gray")
    axes[0].set_title(f"Input 224×224\nGT = {gt}   p = {row['prob']:.3f}")

    for k, (d, cam) in enumerate((("pos", cam_pos), ("neg", cam_neg))):
        vmax = max(np.percentile(cam, pct), 1e-6)
        _overlay(axes[1 + k], img, upsample(cam), vmax, CMAP[d])
        axes[1 + k].set_title(f"{DIR_LABEL[d]}\noverlay", color=DIR_COLOR[d],
                              fontweight="bold")
        axes[3 + k].imshow(cam, cmap=CMAP[d], vmin=0, vmax=vmax)
        # The raw max is what `dominance` is built from, so show it per channel.
        axes[3 + k].set_title(f"{DIR_LABEL[d]}\nraw 7×7  (max {(s_pos if d == 'pos' else s_neg):.4g})",
                              color=DIR_COLOR[d], fontsize=9)

    for ax in axes:
        ax.axis("off")
    quad = row["cell"]
    plt.suptitle(f"{disease}  |  {quad}  |  p={row['prob']:.3f}  t={ths:.3f}  |  "
                 f"dominance = {dom:.2f} ({'pro-disease' if dom > 0.5 else 'pro-healthy'})"
                 f"\n{row['path']}",
                 color=QUAD_COLOR.get(quad, "#444"), fontweight="bold")
    plt.tight_layout()
    plt.show()


def sample(quad, n=1, seed=None):
    """n random cam_files from one quadrant: TP / TN / FP / FN."""
    pool = meta.index[meta["cell"] == quad]
    if not len(pool):
        raise ValueError(f"no rows in quadrant {quad!r}")
    rng = np.random.RandomState(seed)
    return list(rng.choice(pool, size=min(n, len(pool)), replace=False))


print("helpers ready:  show_cam(file)   sample('TP', n)   load_cam -> pos, neg, s_pos, s_neg")
print(f"threshold = {ths:.4f}   alpha: max={ALPHA_MAX} gamma={ALPHA_GAMMA}")
print("colours: pro-disease = red ramp, pro-healthy = blue ramp")

In [ ]:
# ── one image, both directions ───────────────────────────────────────────────
# Every panel row is the SAME image: the pro-disease map and the pro-healthy map
# side by side, so you can read what argued each way on that one X-ray.
#
# What to look for per quadrant:
# TP  model says disease, and it is      -> pro-disease map should sit on the finding
# TN  model says healthy, and it is      -> pro-healthy map broad, pro-disease weak
# FP  model says disease, but it is not  -> what the pro-disease map mistook for it
# FN  model says healthy, but it is not  -> whether the pro-disease map DID find the
#                                           lesion while the pro-healthy one outweighed it
#
# That last case is the one a single map could never show: pre-split, an FN kept
# only the winning direction, so a correctly-localised but outvoted finding was
# indistinguishable from never having been seen.
QUAD = "FN"

for f in sample(QUAD, n=1):
    show_cam(f)

In [ ]:
# ── one example from EACH quadrant, both directions ──────────────────────────
# Four images, each with its two maps. Read down the `dominance` values in the
# titles: it should track the decision (high on TP/FP, low on TN/FN) while the
# MAPS stay informative in both directions regardless -- that separation is what
# the two-channel format buys. Under the old single-map format the TN and FN
# rows were blank, so there was nothing to compare here at all.
SEED = 0

for quad in ["TP", "TN", "FP", "FN"]:
    for f in sample(quad, n=1, seed=SEED):
        show_cam(f)

In [ ]:
# ── grid: N images per quadrant, both directions stacked ─────────────────────
# Two rows per quadrant, eight in total. A COLUMN is one image: the pro-disease
# map directly above the pro-healthy map for that same X-ray, so the pair reads
# vertically and the quadrant reads horizontally.
#
# Good for spotting whether a direction has a systematic shape independent of
# anatomy -- e.g. every pro-healthy map lighting the same region regardless of
# the patient, which would mean that channel encodes a global prior rather than
# the image. The mean/std cell further down tests that properly.
N = 6
SEED = 1
QUADS = ["TP", "TN", "FP", "FN"]

fig, axes = plt.subplots(2 * len(QUADS), N, figsize=(2.3 * N, 4.6 * len(QUADS)))
for qi, quad in enumerate(QUADS):
    files = sample(quad, n=N, seed=SEED)
    for c in range(N):
        for k, d in enumerate(("pos", "neg")):
            ax = axes[2 * qi + k, c]
            ax.axis("off")
            if c >= len(files):
                continue
            row = meta.loc[files[c]]
            cam_pos, cam_neg, s_pos, s_neg = load_cam(files[c])
            cam = cam_pos if d == "pos" else cam_neg
            img = load_xray(row["path"])
            _overlay(ax, img, upsample(cam),
                     max(np.percentile(cam, 99.0), 1e-6), CMAP[d])
            # Title only the top row of each pair: p and dominance are properties
            # of the image, not of the direction, so repeating them would imply
            # the two panels are different images.
            if k == 0:
                ax.set_title(f"p={row['prob']:.3f}  d={dominance(s_pos, s_neg):.2f}",
                             fontsize=8, color=QUAD_COLOR[quad])

    # axis is off, so label the rows with text instead of set_ylabel
    for k, d in enumerate(("pos", "neg")):
        axes[2 * qi + k, 0].text(
            -0.12, 0.5, f"{quad}\n{DIR_LABEL[d]}",
            transform=axes[2 * qi + k, 0].transAxes, rotation=90,
            va="center", ha="center", fontsize=9, fontweight="bold",
            color=DIR_COLOR[d])

plt.suptitle(f"{disease} — Grad-CAM, both directions, by confusion quadrant  "
             f"(t = {ths:.3f};  d = pro-disease share of raw gradient)",
             fontsize=13, y=0.997)
plt.tight_layout()
plt.show()

## Emptiness and balance, per direction

Two questions the single-map format could not separate:

1. **Is either channel blank?** A direction that is all-zero on a quadrant means
   the split did not actually recover it. This is the check the old format failed:
   77.5% of TN and 50.0% of FN were exactly empty.
2. **How lopsided is the raw gradient?** Both channels are normalised on disk, so
   `dominance = scale_pos / (scale_pos + scale_neg)` is the only surviving record
   of which direction actually won, and by how much.

In [ ]:
# `cell` and `pred` are assigned once, up in the metadata cell. Uncertain (-1)
# rows sit in their own "uncertain" bucket and are excluded here.
#
# Reading 95,835 npz files takes a few minutes; SUBSAMPLE caps it per quadrant.
SUBSAMPLE = 3000          # None = every row
FLAT_STD_THRESH = 0.05    # a CAM with std below this looks visually uniform

quad = meta[meta["cell"].isin(["TP", "TN", "FP", "FN"])]
if SUBSAMPLE:
    quad = quad.groupby("cell", group_keys=False).apply(
        lambda g: g.sample(min(len(g), SUBSAMPLE), random_state=0))

# One row per (image, direction): stats are per-channel, so a long frame keeps
# the two directions comparable with a single groupby instead of parallel columns.
stats = []
for f, r in quad.iterrows():
    cam_pos, cam_neg, s_pos, s_neg = load_cam(f)
    dom = dominance(s_pos, s_neg)
    for d, cam, scale in (("pos", cam_pos, s_pos), ("neg", cam_neg, s_neg)):
        stats.append(dict(cell=r["cell"], dirn=d, prob=r["prob"], dominance=dom,
                          nnz=int(np.count_nonzero(cam)), cam_std=float(cam.std()),
                          scale=scale))
st = pd.DataFrame(stats)
st["empty_exact"] = st["nnz"] == 0
st["empty_flat"] = st["cam_std"] < FLAT_STD_THRESH

n_img = len(st) // 2
print(f"{disease}   ({n_img:,} images x 2 directions"
      f"{f', subsampled to {SUBSAMPLE:,}/quadrant' if SUBSAMPLE else ''})\n")

summary = st.groupby(["cell", "dirn"]).agg(
    n=("cell", "size"),
    empty_exact_rate=("empty_exact", "mean"),
    empty_flat_rate=("empty_flat", "mean"),
    mean_cam_std=("cam_std", "mean"),
    mean_scale=("scale", "mean"),
).reindex(pd.MultiIndex.from_product([["TP", "TN", "FP", "FN"], ["pos", "neg"]],
                                     names=["cell", "dirn"]))
print(summary.round(4).to_string())

# dominance is per IMAGE, so take it off one direction's rows only -- averaging
# over the long frame would just double-count every image.
dom_by_cell = st[st["dirn"] == "pos"].groupby("cell")["dominance"]
print("\ndominance (pro-disease share of the raw gradient), by quadrant:")
print(dom_by_cell.agg(mean="mean", median="median",
                      frac_pro_disease=lambda s: (s > 0.5).mean())
      .reindex(["TP", "TN", "FP", "FN"]).round(4).to_string())

print(f"\noverall exact-empty: {st['empty_exact'].mean():.1%}"
      f"   flat (std<{FLAT_STD_THRESH}): {st['empty_flat'].mean():.1%}")

# The bar to clear: BOTH directions non-empty in every quadrant, with comparable
# mean_cam_std. "Not blank" and "carries structure" are different bars, and a
# merely-non-zero map would show a much lower std. Under the old single-map
# format this plot had one bar per quadrant and TN/FN were at 0.775/0.500.
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for a, (col, ylab, title) in zip(ax[:2], [
        ("empty_exact_rate", "fraction of all-zero CAMs", "Empty-CAM rate"),
        ("mean_cam_std", "mean CAM std", "Mean CAM spread")]):
    summary[col].unstack("dirn")[["pos", "neg"]].plot.bar(
        ax=a, rot=0, color=[DIR_COLOR["pos"], DIR_COLOR["neg"]])
    a.set_ylabel(ylab)
    a.set_title(f"{title} by quadrant × direction")
    a.set_xlabel("")
    a.legend([DIR_LABEL["pos"], DIR_LABEL["neg"]], fontsize=8)
ax[0].set_ylim(0, 1)

# Where the decision actually lives: dominance either side of 0.5.
for q in ["TP", "TN", "FP", "FN"]:
    ax[2].hist(dom_by_cell.get_group(q), bins=40, range=(0, 1), histtype="step",
               lw=1.8, label=q, color=QUAD_COLOR[q])
ax[2].axvline(0.5, color="#444", ls="--", lw=1)
ax[2].set_xlabel("dominance  (0 = all pro-healthy, 1 = all pro-disease)")
ax[2].set_ylabel("images")
ax[2].set_title("Raw gradient balance by quadrant")
ax[2].legend(fontsize=8)

plt.suptitle(f"{disease} — Grad-CAM informativeness and balance, by direction", y=1.03)
plt.tight_layout(); plt.show()

## Is a direction's map image-specific, or a constant prior?

The pro-healthy maps look broad and diffuse, which is expected — "nothing is
wrong here" has no single location the way a focal finding does. But a map that is
*the same* on every image would look identical too, and would mean that channel
encodes a global bias rather than anything about this patient.

The mean map separates the two. If the per-image deviation from the mean is small
relative to the mean itself, the maps are a prior, not an explanation. Both
directions get the test separately: one channel can be image-specific while the
other is a constant, and that asymmetry is exactly what a single collapsed map
would hide.

In [ ]:
N_MEAN = 10000   # images per quadrant
QUADS = ["TP", "TN", "FP", "FN"]

# 4 rows: mean and std, for each direction. Columns are quadrants, so a column
# is one quadrant's full profile and a row-pair is one direction's.
fig, axes = plt.subplots(4, len(QUADS), figsize=(15, 14.4))
print(f"{'quad':6s} {'dirn':5s} {'mean|cam|':>10s} {'mean std across imgs':>22s} {'ratio':>8s}")
print("-" * 56)

for c, q in enumerate(QUADS):
    files = sample(q, n=N_MEAN, seed=2)
    # Read each npz once and keep both channels; reading the directory twice
    # would double the I/O on 10k files per quadrant for no reason.
    loaded = [load_cam(f) for f in files]
    stacks = {"pos": np.stack([l[0] for l in loaded]),
              "neg": np.stack([l[1] for l in loaded])}

    for k, d in enumerate(("pos", "neg")):
        arr = stacks[d]
        mean_map = arr.mean(0)
        std_map = arr.std(0)            # spread ACROSS images, per cell

        axes[2 * k, c].imshow(mean_map, cmap=CMAP[d])
        axes[2 * k, c].set_title(f"{q}  mean of {len(files)}", color=QUAD_COLOR[q],
                                 fontweight="bold", fontsize=10)
        axes[2 * k + 1, c].imshow(std_map, cmap="magma")
        axes[2 * k + 1, c].set_title(f"{q}  std across images", color=QUAD_COLOR[q],
                                     fontsize=10)
        for r in (2 * k, 2 * k + 1):
            axes[r, c].axis("off")

        # ratio >~ 1 means images differ from each other as much as the mean
        # signal: the map is image-specific. Near 0 means every image gives the
        # same map.
        ratio = std_map.mean() / (mean_map.mean() + 1e-8)
        print(f"{q:6s} {d:5s} {mean_map.mean():10.4f} {std_map.mean():22.4f} {ratio:8.2f}")

for k, d in enumerate(("pos", "neg")):
    for r, lab in ((2 * k, "mean"), (2 * k + 1, "std")):
        axes[r, 0].text(-0.18, 0.5, f"{DIR_LABEL[d]}\n{lab}",
                        transform=axes[r, 0].transAxes, rotation=90,
                        va="center", ha="center", fontsize=10,
                        fontweight="bold", color=DIR_COLOR[d])

plt.suptitle(f"{disease} — average CAM and per-image variability, by quadrant × direction",
             fontsize=13, y=0.997)
plt.tight_layout()
plt.show()

print("\nratio ~1 or above: maps vary image to image (a real explanation).")
print("ratio near 0:      every image yields the same map (a global prior).")
print("Compare the two directions per quadrant: a low ratio on `neg` alone would")
print("mean the pro-healthy channel is a fixed template, not patient evidence.")